# P3: Houghton Weather — Snow, Heat, and Rain

### The application
A campus planning team wants an evidence-based description of Houghton's weather to guide seasonal maintenance and outdoor-event preparation. Your task is to combine **Michigan Tech's published snowfall tables** with **daily historical weather from Open-Meteo**, summarize snow, heat, and heavy-rain conditions, and explain what the data can support.

**Study period:** January 1, 2019–December 31, 2024. Compare snowfall only for complete **October–May** seasons with data in both sources. Use the six calendar years for heat and rain summaries.

**Final products:** yearly weather indicators, aligned winter snowfall comparisons, two figures, and a short planning brief. This is a descriptive data-integration project; six years do not establish a long-term climate trend.


### Data and units
| Source | What it contributes | Interpretation |
|---|---|---|
| Michigan Tech snowfall webpage | Local October–May snowfall table entries, in inches | Published local records; some cells are blank or incomplete |
| Open-Meteo Historical Weather API, ERA5 | Daily maximum/minimum temperature, rainfall, and snowfall near Houghton | Gridded reanalysis estimates, not a campus snow gauge |

Temperatures are **°C**, rain is **mm**, and API snowfall is **cm of snowfall depth**. Convert local snowfall with **1 inch = 2.54 cm**. Do not add rain in mm to snowfall depth in cm, and do not treat snowfall depth as liquid-water equivalent.

### Working rules
- Keep answers in the designated cells. Loops and helper functions are allowed; reuse earlier functions.
- Each question handles one manageable stage. Request helpers and calendar-day calculations are supplied.
- The notebook embeds real source snapshots for reproducibility. Default grading is offline; no API key is needed. Optional live mode uses the same HTTP/JSON workflow.
- Missing snowfall cells mean **unknown**, not zero. A published total alone does not prove that a season is complete.
- Do not hard-code source values, row counts, seasons, or final findings. The auto questions include two public and two hidden checks each.
- Thresholds below are project definitions for comparison, not official weather-warning criteria.


In [ ]:
import re
import json
import gzip
import base64
import copy
import datetime
from unittest.mock import Mock, patch

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import otter
grader = otter.Notebook()

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

LIVE_MODE = False
SNOW_URL = "https://www.mtu.edu/alumni/favorites/snowfall/"
API_URL = "https://archive-api.open-meteo.com/v1/archive"
DAILY_VARIABLES = "temperature_2m_max,temperature_2m_min,rain_sum,snowfall_sum"
MONTH_COLUMNS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May"]
RAW_SNOW_COLUMNS = ["Winter"] + MONTH_COLUMNS + ["Total_inches"]
DAILY_COLUMNS = ["Date", "Tmax_c", "Tmin_c", "Rain_mm", "Snow_cm"]
WEB_COLUMNS = ["Start_year", "Web_snow_cm", "Web_complete"]
YEAR_COLUMNS = ["Year", "Days", "Hot_days", "Heavy_rain_days", "Heavy_snow_days", "Rain_mm", "Snow_cm", "Max_temp_c"]
WINTER_COLUMNS = ["Start_year", "Days", "Expected_days", "API_snow_cm"]
COMPARISON_COLUMNS = ["Start_year", "API_snow_cm", "Web_snow_cm", "Difference_cm", "Difference_pct"]

---
<a id="outline"></a>

## Question Navigation

- [Q1. Scrape the snowfall tables](#q1)
- [Q2. Clean winter labels and snowfall units](#q2)
- [Q3. Request the historical weather API](#q3)
- [Q4. Convert JSON arrays to a table](#q4)
- [Q5. Clean and validate daily records](#q5)
- [Q6. Create weather-event indicators](#q6)
- [Q7. Summarize calendar-year conditions](#q7)
- [Q8. Calculate complete winter snowfall](#q8)
- [Q9. Compare snowfall sources fairly](#q9)
- [Q10. Build a snow–heat–rain dashboard](#q10)
- [Q11. Visualize the snowfall comparison](#q11)
- [Q12. Write the campus weather brief](#q12)

### Provided snapshots — run unchanged
The collapsed cell contains the original captured webpage and API JSON. The readable request helpers below load those responses. You do not need to write storage or network-retry code.


In [ ]:
_SNAPSHOT_B64 = '...'
SNAPSHOT = json.loads(gzip.decompress(base64.b64decode(_SNAPSHOT_B64)))
del _SNAPSHOT_B64


In [ ]:
REQUEST_PARAMS = SNAPSHOT["provenance"]["params"]

class SnapshotResponse:
    """Provided response interface for offline requests."""
    def __init__(self, text=None, payload=None):
        self.text = text
        self.payload = payload
        self.status_code = 200
    def raise_for_status(self):
        return None
    def json(self):
        return copy.deepcopy(self.payload)

_REQUEST_CACHE = {}

def weather_get(url, params=None, timeout=20):
    """Provided: use saved responses by default; real requests in live mode."""
    if not LIVE_MODE:
        if url == SNOW_URL:
            return SnapshotResponse(text=SNAPSHOT["snow_html"])
        if url == API_URL and params == REQUEST_PARAMS:
            return SnapshotResponse(payload=SNAPSHOT["weather"])
        raise ValueError("The offline snapshot supports only the specified study request.")
    cache_key = (url, tuple(sorted((params or {}).items())))
    if cache_key not in _REQUEST_CACHE:
        response = requests.get(url, params=params, timeout=timeout)
        response.raise_for_status()
        response.encoding = "utf-8"
        _REQUEST_CACHE[cache_key] = response
    return _REQUEST_CACHE[cache_key]

def expected_winter_days(start_year):
    """Provided: October 1 through May 31, including leap day when present."""
    return (datetime.date(int(start_year)+1, 5, 31) - datetime.date(int(start_year), 10, 1)).days + 1

snow_response = weather_get(SNOW_URL, timeout=20)
snow_response.raise_for_status()
snow_html = snow_response.text
print("Snapshot captured (UTC):", SNAPSHOT["provenance"]["retrieved_utc"])
print("Requested coordinates:", REQUEST_PARAMS["latitude"], REQUEST_PARAMS["longitude"])
print("Study:", REQUEST_PARAMS["start_date"], "to", REQUEST_PARAMS["end_date"])


---
### Part A: Scrape and clean local snowfall records
The webpage contains multiple snowfall tables. We use the first two matching tables to cover the recent period and the previous decade. Their first heading is `Winter` and last heading is `Total Inches`. Older rows sometimes use `th` for the winter label and `td` for measurements.


<a id="q1"></a>

## Q1. Scrape the snowfall tables — 12 points (auto)

[Return to outline](#outline)

Write `scrape_snow(html)`. Find tables whose `thead th` headings begin with `Winter` and end with `Total Inches`. From the **first two** matching tables, extract each `tbody tr` using both `th` and `td` cells. Keep rows with ten cells and return `RAW_SNOW_COLUMNS` in source order.

Use BeautifulSoup and `.get_text(strip=True)`. Retain raw strings and blank cells; do not convert numbers or filter years yet. An HTML page with no matching table returns the same empty schema.


In [ ]:
def scrape_snow(html):
    soup = ...
    tables = ...
    rows = []
    for table in ...:
        for row in ...:
            values = ...
            if ...:
                rows.append(...)
    return pd.DataFrame(..., columns=...)

raw_snow = scrape_snow(snow_html)
raw_snow.head()

In [ ]:
grader.check("q1")

[↑ Return to outline](#outline)


<a id="q2"></a>

## Q2. Clean snowfall data — 12 points (auto)

[Return to outline](#outline)

Write `clean_web_snow(raw)` to:

1. Extract the four-digit starting year from `Winter` using regex. Keep years **2019–2023**.
2. Convert `MONTH_COLUMNS` and `Total_inches` to numeric values: remove commas, then extract the first number. Leave blanks and nonnumeric values missing.
3. Create `Web_complete`: `True` when all eight months and the total are present.
4. Create `Web_snow_cm` by multiplying `Total_inches` by **2.54**.

Return only `WEB_COLUMNS`, sorted by integer `Start_year`, with a reset index. Keep incomplete seasons and do not replace missing values with zero.

In [ ]:
def clean_web_snow(raw):
    result = raw.copy()

    result["Start_year"] = ...
    result = ...
    result["Start_year"] = ...

    snow_columns = MONTH_COLUMNS + ["Total_inches"]
    for column in snow_columns:
        text = ...
        result[column] = ...

    result["Web_complete"] = ...
    result["Web_snow_cm"] = ...

    return ...


web_snow = clean_web_snow(raw_snow)
web_snow.head()

In [ ]:
grader.check("q2")

[↑ Return to outline](#outline)


### Part B: Retrieve and prepare daily weather
The API response stores parallel daily arrays inside `daily`, with unit labels in `daily_units`. We explicitly request **ERA5** for a consistent model choice, Houghton's approximate coordinates, and local calendar days (`America/Detroit`). The returned grid-cell coordinates may differ from the requested point.


<a id="q3"></a>

## Q3. Request the historical weather API — 12 points (auto)

[Return to outline](#outline)

Complete `fetch_weather(start_date, end_date)`. Use the provided parameter scaffold; fill the two dates and `daily` variable list (`DAILY_VARIABLES`). Call `weather_get(API_URL, params=params, timeout=20)`, check status with `.raise_for_status()`, and return `.json()`.

The saved response supports `2019-01-01` through `2024-12-31`; hidden tests substitute a response to check other dates. Do not hide request errors as empty data. No date-validation function or API key is required.


In [ ]:
def fetch_weather(start_date, end_date):
    params = {"latitude": 47.1219, "longitude": -88.569,
              "start_date": ..., "end_date": ..., "daily": ...,
              "timezone": "America/Detroit", "models": "era5"}
    response = ...
    ...  # Check HTTP status.
    return ...


weather_json = fetch_weather("2019-01-01", "2024-12-31")
weather_json["daily_units"]

In [ ]:
grader.check("q3")

[↑ Return to outline](#outline)


<a id="q4"></a>

## Q4. Convert JSON arrays to a table — 10 points (auto)

[Return to outline](#outline)

Write `daily_to_frame(payload)`. Build a DataFrame from `payload['daily']` and rename:
`time → Date`, `temperature_2m_max → Tmax_c`, `temperature_2m_min → Tmin_c`, `rain_sum → Rain_mm`, `snowfall_sum → Snow_cm`.

Return exactly `DAILY_COLUMNS` in source order. Keep dates as strings for now and preserve missing values. Assume all five arrays exist and have equal length. This function should take only a few lines.


In [ ]:
def daily_to_frame(payload):
    ...


raw_daily = daily_to_frame(weather_json)
raw_daily.head()

In [ ]:
grader.check("q4")

[↑ Return to outline](#outline)


<a id="q5"></a>

## Q5. Clean daily weather data — 12 points (auto)

[Return to outline](#outline)

Write `clean_daily(raw)` to:

1. Copy the input.
2. Convert `Date` to datetime and the four measurements to numeric using `errors="coerce"`.
3. Remove duplicate dates, keeping the first row, then drop rows with missing values in `DAILY_COLUMNS`.
4. Remove rows where `Tmax_c < Tmin_c`, rain is negative, or snow is negative.
5. Return `DAILY_COLUMNS`, sorted by date with a reset index.

Negative temperatures are valid.

In [ ]:
def clean_daily(raw):
    result = raw.copy()
    ...


daily = clean_daily(raw_daily)
print("Retained days:", len(daily))

In [ ]:
grader.check("q5")

[↑ Return to outline](#outline)


### Part C: Build indicators and align winter seasons
Use these operational definitions consistently: **hot day = Tmax ≥ 30°C**, **heavy-rain day = rain ≥ 20 mm**, **heavy-snow day = snowfall ≥ 10 cm**. These are transparent analysis thresholds, not official alerts. A day can satisfy more than one condition.


<a id="q6"></a>

## Q6. Add weather indicators — 14 points (auto)

[Return to outline](#outline)

Write `add_indicators(daily)` to return a copy with these new columns:

- `Year` and `Month`: extracted from `Date`.
- `Hot`: `True` when `Tmax_c >= 30`.
- `Heavy_rain`: `True` when `Rain_mm >= 20`.
- `Heavy_snow`: `True` when `Snow_cm >= 10`.
- `Start_year`: the winter's starting year:
  - October–December: current year.
  - January–May: previous year.
  - June–September: `pd.NA`.

Use `"Int64"` for `Start_year` and keep the original row order.

In [ ]:
def add_indicators(daily):
    ...


features = add_indicators(daily)
features.head()

In [ ]:
grader.check("q6")

[↑ Return to outline](#outline)


<a id="q7"></a>

## Q7. Summarize yearly weather — 14 points (auto)

[Return to outline](#outline)

Write `summarize_years(features)` to group the records by `Year` and create a DataFrame with:

- `Days`: number of records in each year.
- `Hot_days`: sum of `Hot`.
- `Heavy_rain_days`: sum of `Heavy_rain`.
- `Heavy_snow_days`: sum of `Heavy_snow`.
- `Rain_mm`: sum of `Rain_mm`.
- `Snow_cm`: sum of `Snow_cm`.
- `Max_temp_c`: maximum `Tmax_c`.

Reset the index so `Year` becomes a column. Return `YEAR_COLUMNS` in the specified order, with years sorted ascending. Keep incomplete years.

In [ ]:
def summarize_years(features):
    ...


annual = summarize_years(features)
annual

In [ ]:
grader.check("q7")

[↑ Return to outline](#outline)


<a id="q8"></a>

## Q8. Calculate winter snowfall — 14 points (auto)

[Return to outline](#outline)

Write `summarize_winters(features)` to:

1. Remove rows with missing `Start_year`.
2. Group by `Start_year` and calculate:
   - `Days`: number of daily records.
   - `API_snow_cm`: total `Snow_cm`.
3. Create `Expected_days` using the provided `expected_winter_days(year)` function.
4. Keep only complete winters where `Days == Expected_days`.

Return `WINTER_COLUMNS`, sorted by `Start_year` with a reset index. A winter runs from October through May.

In [ ]:
def summarize_winters(features):
    ...


winters = summarize_winters(features)
winters

In [ ]:
grader.check("q8")

[↑ Return to outline](#outline)


<a id="q9"></a>

## Q9. Compare snowfall sources — 12 points (auto)

[Return to outline](#outline)

Write `compare_snow(winters, web_snow)` to:

1. Keep web records where `Web_complete` is `True`.
2. Inner join the two DataFrames on `Start_year`.
3. Calculate:
   - `Difference_cm = API_snow_cm - Web_snow_cm`
   - `Difference_pct = 100 * Difference_cm / Web_snow_cm`
4. Return `COMPARISON_COLUMNS`, sorted by `Start_year` with a reset index.

Assume unique season keys and positive web snowfall totals. A negative difference means the API estimate is lower.

In [ ]:
def compare_snow(winters, web_snow):
    ...


comparison = compare_snow(winters, web_snow)
comparison

In [ ]:
grader.check("q9")

[↑ Return to outline](#outline)


### Part D: Communicate findings for campus planning — 37 manual points


<!-- BEGIN QUESTION -->

<a id="q10"></a>

## Q10. Build a snow–heat–rain dashboard — 15 points (manual)

[Return to outline](#outline)

Create one figure with **three panels** using your computed tables:
1. `annual`: hot-day count by calendar year.
2. `annual`: rainfall total (mm) by calendar year.
3. `winters`: API snowfall total (cm) by complete October–May season.

Use bar charts or line plots. Label units and clearly distinguish calendar years from winter seasons (e.g., `2020–21`). Give each panel a title. Keep different units on separate axes. No written response is required.

**Rubric:** correct heat, rain, and winter-snow series (7), titles and labels with correct units/season definitions (4), readable three-panel layout (4).


In [ ]:
# Create your figure here.
...

<!-- END QUESTION -->

[↑ Return to outline](#outline)


<!-- BEGIN QUESTION -->

<a id="q11"></a>

## Q11. Visualize the snowfall comparison — 12 points (manual)

[Return to outline](#outline)

Using `snow_comparison`, create a **grouped bar chart** showing API and published local snowfall side by side for each matched complete winter. Use centimeters for both sources, winter labels such as `2020–21`, a title, axis labels, and a legend distinguishing **ERA5 estimate** from **published local record**. Plot only the aligned seasons from Q9. No written response is required.

**Rubric:** correct paired values and complete matched winters (6), shared centimeter units and accurate season labels (2), title/axes and source legend (2), readable layout (2).


In [ ]:
snow_comparison = compare_snow(winters, web_snow)
# Create your figure here.
...

<!-- END QUESTION -->

[↑ Return to outline](#outline)


<!-- BEGIN QUESTION -->

<a id="q12"></a>

## Q12. Write the campus weather brief — 10 points (manual)

[Return to outline](#outline)

Write **120–180 words** for the campus planning team. Include one numerical finding each about **snow, heat, and rain**, then propose one practical planning priority supported by your output. Explain why the two snowfall sources can disagree, and acknowledge incomplete records and the short study period. Do not claim that six years prove a climate trend or predict a particular future storm.

Use your actual computed results; there is no required “correct” priority.

**Rubric:** correct numerical findings covering snow, heat, and rain (4), explanation of source differences and missing-record handling (3), supported planning priority and appropriate limits on inference (3).


_Type your answer here, replacing this text._

**Your answer:** Replace this text with your response...


<!-- END QUESTION -->

[↑ Return to outline](#outline)


### Sources, provenance, and optional live use
- [Michigan Tech Snowfall Records](https://www.mtu.edu/alumni/favorites/snowfall/): original HTML, including blank monthly cells. The page states that its Keweenaw Research Center no longer measures daily snowfall; the site should not be assumed to be one unchanging sensor or location across years.
- [Open-Meteo Historical Weather API documentation](https://open-meteo.com/en/docs/historical-weather-api): ERA5 reanalysis, daily maximum/minimum temperature (°C), rain (mm), and snowfall (cm). Source data: ERA5, ECMWF/Copernicus Climate Change Service; API service: Open-Meteo. Keep attribution when sharing derived figures.
- The exact request parameters and capture time are embedded and printed above. Daily values are real API output, not generated weather. Small boundary cases in tests are synthetic.

**Optional live check (ungraded):** set `LIVE_MODE=True` and rerun once. The helper checks HTTP status and caches successful responses; stop on errors/rate limits and retry later rather than repeatedly running everything. Live table contents and archive values may change. Restore `LIVE_MODE=False`, restart the kernel, and run all cells for the graded export.

## Congratulations! You have finished P3! 

### Instructions
Use the generated student notebook. Run all cells in order, check Q1–Q9, and save both figures and the Q12 brief. Run Otter's generated export cell and submit the ZIP to Gradescope. Default tests use the embedded snapshots, so a live service outage does not affect grading.


## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)